# Fit inspection

Visual QA of a fitted spectrum plus a parameter table with uncertainties.
All functions live in `alibz.inspection` and are exported from `alibz`:

| Function | Purpose |
|---|---|
| `plot_spectrum_overview(x, y, fit_dict, xlim=..., log_scale=...)` | raw + background, fit + peak locations, residual vs ±2σ local-noise band |
| `plot_peak_zoom(x, y, fit_dict, center_nm, span_nm=...)` | one peak: data, own component, other components, residual, parameters ± σ |
| `peak_table(x, y, fit_dict)` / `format_peak_table(rows)` | per-peak center/area/σ/γ ± 1σ, FWHM, height, S/N |
| `estimate_peak_uncertainties(x, y_bgsub, peaks)` | the underlying joint-GLS blend-group σ's |

**Reading the uncertainties:** blended peaks carry honestly inflated σ
(joint covariance per blend group); parameters at a zero bound show
`(pinned)`; for strong isolated lines σ is a statistical lower bound —
the pipeline's empirical scatter can exceed it several-fold (documented
in the docstring and pinned by `tests/test_inspection.py`).

In [ ]:
import numpy as np
from alibz import (PeakyFinder, peak_table, format_peak_table,
                   plot_spectrum_overview, plot_peak_zoom)

PATH = '/Users/mwhittaker/Library/CloudStorage/GoogleDrive-mwhittaker@lbl.gov/My Drive/Postdocs/Xuan Cao/Data/LIBS/MW2-112/raw/'
SPECTRUM = 0

finder = PeakyFinder(PATH)
finder.data.load_data()
x, y = finder.data.get_data()[SPECTRUM]
fit = finder.fit_spectrum(x, y, subtract_background=True, plot=False, n_sigma=0)
print(f"{fit['sorted_parameter_array'].shape[0]} peaks fitted")

## Full-spectrum overview
Raw + background, fitted model with peak locations, and the residual against the ±2σ local-noise band (structure outside the band = real misfit).

In [ ]:
fig, axs = plot_spectrum_overview(x, y, fit)

## Zoom into a region
Same three panels, windowed — e.g. the Ca II H&K region.

In [ ]:
fig, axs = plot_spectrum_overview(x, y, fit, xlim=(392.5, 398.5))

## Single-peak inspection
Data, this peak's Voigt component, everything else, and the residual —
with fitted parameters ± uncertainties in the title.  On Li I 670.8 the
S-shaped residual outside the noise band is the self-absorption
signature (flat-topped core, suppressed wings).

In [ ]:
fig, axs = plot_peak_zoom(x, y, fit, 670.78, span_nm=2.5)

## Parameter table with uncertainties
Strongest peaks first (`sort_by="center"` for wavelength order).  Note
the K I 766.5 split components carrying honestly blend-inflated errors,
and `(pinned)` marking bound-constrained parameters.

In [ ]:
rows = peak_table(x, y, fit)
print(format_peak_table(rows, max_rows=25))

The rows are plain dicts, so filtering and export are one-liners:

```python
suspicious = [r for r in rows if r["snr"] < 15 or r["fwhm_nm"] > 0.5]
import csv
with open("peaks.csv", "w") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
```